# الدرس 18: تأمين وكلاء الذكاء الاصطناعي باستخدام إيصالات التشفير

## دفتر التمارين العملي

يشرح هذا الدفتر أربع تمارين:

1. **توقيع إيصالك الأول** لمكالمة أداة الوكيل والتحقق منه.
2. **التلاعب بالإيصال** ومراقبة فشل التحقق.
3. **بناء سلسلة من ثلاثة إيصالات** وتأكيد سلامة السلسلة.
4. **تغليف مكالمة أداة إطار عمل وكيل مايكروسوفت** بحيث يصدر كل إجراء إيصالًا.

جميع البدائيات التشفيرية مستوردة من مكتبات مُدارة جيدًا (`pynacl` لـ Ed25519، `jcs` لـ RFC 8785 JSON المنسوخ، `hashlib` من مكتبة بايثون القياسية لـ SHA-256). منطق الإيصال نفسه هو بايثون بسيط يمكنك قراءته وتعديله.

قم بتشغيل الخلايا بالترتيب. كل قسم قصير ومكتفي بذاته.


## الإعداد

قم بتثبيت الاعتماديات الاثنين. كلاهما يحمل رخصاً متساهلة (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## أدوات المساعدة

هذان الأداتان تساعدان في ترميز base64url (بدون الحشو) وتجزئة SHA-256 للكائنات العشوائية. تحافظان على تركيز بقية الدفتر على منطق الإيصال نفسه.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## القسم 1: قم بتوقيع إيصالك الأول

تخيل أن وكيلنا لشركة **Contoso Travel** قد بحث للتو عن رحلات من سيدني إلى لوس أنجلوس لعميل. نريد تسجيل استدعاء هذه الأداة كإيصال موقع حتى يتمكن مدقق مستقبلي من التحقق منه دون الحاجة للثقة بنا.

### الخطوة 1.1: إنشاء مفتاح توقيع

في بيئة الإنتاج، سيعيش مفتاح توقيع الوكيل في وحدة أمان الأجهزة (HSM)، أو Azure Key Vault، أو مخزن محمي مشابه. في هذا الدرس، نقوم بإنشاء مفتاح جديد في الذاكرة.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### الخطوة 1.2: بناء حمولة الإيصال

تحتوي الحمولة على كل شيء نريد أن يشهد عليه الإيصال: من قام بالفعل، وأي أداة، وبأي وسائط، وما تم إرجاعه، وبأي سياسة، ومتى. نقوم بتجزئة الوسائط والنتيجة بدلاً من تضمينها مباشرة حتى لا يكشف الإيصال محتوى حساسًا.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### الخطوة 1.3: توقيع وتجميع الإيصال

ثلاث خطوات:

1. تحويل الحمولة إلى الشكل القانوني باستخدام JCS بحيث تنتج تطبيقات مختلفة نفس الإيصال المنطقي بطريقة تنتج نفس البايتات بالضبط.
2. تجزئة البايتات القانونية بواسطة SHA-256.
3. توقيع التجزئة بمفتاح Ed25519 الخاص.

ثم يتم إرفاق التوقيع مع الحمولة الأصلية لإنتاج الإيصال النهائي.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### الخطوة 1.4: التحقق من الإيصال

التحقق يعكس العملية. نقوم بإزالة التوقيع، ونعيد حساب التجزئة القانونية، ونفحص التوقيع مقابل المفتاح العام الموجود في الإيصال.

المدقق الذي يقوم بهذا التحقق لا يحتاج منا سوى الإيصال نفسه. لا حاجة لاستدعاء خدمة، ولا للاستعلام عن دليل المفاتيح، ولا للثقة.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

يجب أن ترى `Receipt is valid: True`. لقد أنتج الوكيل أول سجل تدقيق موقّع تشفيرياً له.


## القسم 2: التلاعب بالإيصال

النقطة الأساسية من الإيصالات هي أنها تكشف عن حدوث تلاعب. دعونا نثبت ذلك.

سنقوم بتعديل حرف واحد فقط في الإيصال ونشاهد فشل التحقق.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### ماذا حدث لتوه؟

عندما قمنا بتغيير `policy_id`، تغيرت البايتات المعيارية. وتغيرت قيمة هاش SHA-256 لتلك البايتات. التوقيع (الذي كان على الهاش الأصلي) لم يعد يطابق الهاش الجديد. تُعيد عملية التحقق النتيجة `False` بشكل صحيح.

لا توجد طريقة لتعديل أي حقل من الإيصال مع إمكانية التحقق منه، ما لم يكن لدى المهاجم المفتاح الخاص. طالما كان المفتاح الخاص في خزنة مفاتيح وتم نشر المفتاح العام، فإن التلاعب من المستحيل إخفاؤه.

جرب بنفسك: عدّل `tool_name` أو `agent_id` أو `timestamp` في الخلية أعلاه وأعد التشغيل. كل تغير ينتج إيصال غير صالح.


## القسم 3: ربط الإيصالات معًا

تحمي إيصال واحد إجراءً واحدًا. يقوم معظم الوكلاء باتخاذ العديد من الإجراءات. لجعل التسلسل الكامل قابلاً للكشف عن التلاعب، نقوم بربط كل إيصال بالإيصال السابق عن طريق تضمين تجزئة الإيصال السابق في حمولة الإيصال الجديد.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

إذا قام أي شخص بإزالة إيصال أو إعادة ترتيبه، تنكسر السلسلة عند تلك النقطة بالضبط. تفشل عملية التحقق من أي إيصال لاحق لأنه `previous_receipt_hash` لم يعد يتطابق مع التجزئة الفعلية لسلفه.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

الآن قم بقطع السلسلة عن طريق التلاعب بالإيصال الأوسط وأعد التحقق. يفشل الإيصال المتلاعب به في فحص التوقيع الخاص به، ويفشل الإيصال التالي في فحص ارتباط السلسلة الخاص به (لأن `previous_receipt_hash` لم يعد يتطابق مع تجزئة الإيصال الأوسط المعدل).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

الإيصال 0 لا يزال يتحقق (لم يتم تعديله وليس له إيصال سابق يعتمد عليه). الإيصال 1 يفشل في فحص التوقيع الخاص به لأننا غيرنا `tool_args_hash`. الإيصال 2 يفشل في فحص ارتباط السلسلة لأنه تم حساب `previous_receipt_hash` الخاص به بناءً على الإيصال 1 الأصلي (المعدل الآن).

حتى إذا أعاد المهاجم توقيع الإيصال 1 المعدل (وهو ما لا يمكنه فعله بدون المفتاح الخاص)، فإن عدم تطابق ارتباط السلسلة في الإيصال 2 سيكشف التلاعب. لإخفاء التغيير، يجب على المهاجم إعادة توقيع كل إيصال من نقطة التعديل فصاعدًا، الأمر الذي يتطلب امتلاك المفتاح الخاص.


## القسم ٤: تغليف استدعاء أداة الوكيل بتوقيع الإيصال

في النشر الفعلي، لا تريد من كل مؤلف وكيل أن يتذكر استدعاء `make_receipt`. تريد أن يكون توقيع الإيصال تلقائيًا لكل استدعاء أداة.

إليك أبسط نموذج: فئة تغليف تأخذ أي دالة أداة قابلة للاستدعاء وتُرجع نسخة تُصدر إيصالًا منها. هذا يتكيف مع أي إطار وكيل، بما في ذلك إطار وكيل مايكروسوفت (`agent_framework.azure`).

إذا لم يكن لديك مشروع Azure AI Foundry مضبوط، فإن النموذج الوهمي المحلي أدناه لا يزال يوضح النموذج.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### التكامل مع إطار عمل الوكيل من مايكروسوفت

الغلاف `ReceiptedTool` أعلاه مستقل عن الإطار. لاستخدامه داخل وكيل مبني باستخدام إطار عمل الوكيل من مايكروسوفت، سجّل الدالة المغلفة كأداة. إليك مخطط (ستستبدل النموذج بأداة تسجيل حقيقية من Azure AI Foundry):

```python
# الشيفرة الزائفة تظهر شكل التكامل.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# المزود = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# الوكيل = المزود.create_agent(
#     التعليمات="أنت وكيل سفر كونتوسو ...",
#     الأدوات=[receipted_lookup],   # الأداة المغلفة، ليست الدالة الخام
# )
# الاستجابة = الوكيل.run("ابحث عن رحلات من سيدني إلى لوس أنجلوس في يونيو.")
#
# # بعد التشغيل، كل استدعاء لأداة قام به الوكيل يحتوي على إيصال موقع:
# سلسلة التدقيق = receipted_lookup.receipts
```

لا يحتاج إطار العمل الخاص بالوكيل إلى معرفة أي شيء عن الإيصالات. توقيع الإيصال مغلف حول الأداة، وليس مدمجًا في الإطار. هذه هي الطريقة التي تضيف بها الأصل إلى كود الوكيل الحالي دون إعادة كتابة الوكيل.


## ملخص وتحدي التمدد

لقد قمت بـ:

- توليد زوج مفاتيح Ed25519.
- بناء وتوقيع إيصال لاستدعاء أداة الوكيل.
- التحقق من الإيصال دون اتصال باستخدام المفتاح العام فقط.
- العبث بإيصال وملاحظة فشل التحقق.
- بناء سلسلة متسلسلة بالتجزئة لثلاث إيصالات.
- العبث بوسط السلسلة وملاحظة فشل كل من التوقيع ورابط السلسلة.
- تغليف دالة الأداة بتوقيع الإيصال التلقائي.

**تحدي التمدد.** قم بتوسيع مخطط الإيصال بحقل `request_id` (UUID للتتبع الموزع). حدّث `make_receipt` ليشمله، وتأكد من أن الإيصالات لا تزال تتحقق بشكل كامل. ثم عدّل الحقل بعد التوقيع وتأكد من فشل التحقق. هذا يجبرك على استيعاب كيف يساهم كل بايت من الترميز الرسمي في التوقيع.

**حدود هامة.** تثبت الإيصالات ثلاثة أشياء فقط: الإسناد (هذا المفتاح وقع هذا المحتوى)، السلامة (المحتوى لم يتغير منذ التوقيع)، والترتيب (هذا الإيصال جاء بعد ذلك الإيصال). لا تثبت أن إجراء الوكيل كان صحيحًا، أو أن السياسة المسماة في `policy_id` قد تم تقييمها فعليًا، أو أن الوكيل اتبع كل القواعد. الإيصالات هي الأساس. الحوكمة هي النظام الذي تبنيه فوقها.

اقرأ ملف README الخاص بالدرس مرة أخرى مع وضع هذا الحد في الاعتبار. الخطأ الأكثر شيوعًا الذي ترتكبه الفرق مع الإيصالات هو افتراض أن "لدينا إيصالات" تعني "نحن محكومون." هذا ليس صحيحًا. تجعل الإيصالات سلوك الوكيل قابلاً للتدقيق. لكنها لا تجعله صحيحًا.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
